# LeetCode 970 — Powerful Integers bằng Python

Link đề: https://leetcode.com/problems/powerful-integers/ · Độ khó: **Medium** · Chủ đề: Hash Table, Math, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        ...
```

Ghi chú: đề cho phép trả về **theo thứ tự bất kỳ** (mỗi giá trị xuất hiện tối đa một lần). Vì vậy mọi test trong notebook so sánh sau khi `sorted(...)` và kiểm tra không trùng. Bẫy lớn nhất: khi `x == 1` (hoặc `y == 1`) vòng lặp `while p <= bound: p *= x` **không bao giờ dừng**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm mọi số có dạng `x^i + y^j` (`i, j >= 0`) không vượt quá `bound`, mỗi giá trị một lần.
2. Nắm công cụ Python: `set`, `bytearray` đánh dấu, `bisect`, `heapq`, phép luỹ thừa số nguyên.
3. Đi từ **kiểm tra từng giá trị `v`** → **hai vòng lặp trên các luỹ thừa + set** → **bytearray đánh dấu** → **bisect trên luỹ thừa của `y`** → **heap sinh theo thứ tự tăng**.
4. Hiểu mấu chốt: mỗi cơ số `≥ 2` chỉ có `O(log bound)` luỹ thừa `≤ bound` (≤ 20 vì `2^20 > 10^6`), nên số cặp `≤ 21² = 441`; cơ số `1` chỉ có **một** luỹ thừa khác biệt (`1`).
5. Tránh lỗi kinh điển: vòng lặp vô hạn khi cơ số bằng `1`, bắt đầu luỹ thừa từ `1` thay vì `x^0 = 1` (bỏ sót hoặc dư), dùng `<` thay `<=`, trả kết quả trùng (dùng `list` thay `set`), `break` nhầm vòng ngoài.

## 2. Đề bài & ví dụ

Cho ba số nguyên `x`, `y`, `bound`. Trả về danh sách mọi **số mạnh (powerful integer)** có giá trị `<= bound`. Một số là *powerful* nếu biểu diễn được dưới dạng `x^i + y^j` với `i >= 0`, `j >= 0`. Trả về theo thứ tự bất kỳ; mỗi giá trị xuất hiện tối đa một lần.

**Ví dụ 1:** `x = 2, y = 3, bound = 10` → `[2,3,4,5,7,9,10]`.

**Ví dụ 2:** `x = 3, y = 5, bound = 15` → `[2,4,6,8,10,14]`.

**Ràng buộc:** `1 <= x, y <= 100`; `0 <= bound <= 10^6`.

In [1]:
import bisect
import heapq
import itertools
import random
import time
from typing import List

examples = [((2, 3, 10), [2, 3, 4, 5, 7, 9, 10]), ((3, 5, 15), [2, 4, 6, 8, 10, 14])]
for args, want in examples:
    print(f"x, y, bound = {args} -> {want}")

x, y, bound = (2, 3, 10) -> [2, 3, 4, 5, 7, 9, 10]
x, y, bound = (3, 5, 15) -> [2, 4, 6, 8, 10, 14]


## 3. Kiến thức nền

### 3.1 Số lượng luỹ thừa nhỏ hơn `bound`

Với cơ số `b >= 2`, các luỹ thừa `b^0, b^1, …` tăng nghiêm ngặt và vượt `10^6` sau tối đa `20` bước (`2^20 = 1 048 576 > 10^6`). Vậy mỗi cơ số cho `≤ 20` luỹ thừa `≤ bound` (đếm cả `b^0 = 1`: `≤ 20` phần tử với `2^0..2^19`).

In [2]:
print([2 ** k for k in range(21)][-3:], "-> 2^19 =", 2 ** 19, "<= 10^6 < 2^20 =", 2 ** 20)
print("số luỹ thừa của 2 không vượt 10^6:", sum(1 for k in range(30) if 2 ** k <= 10 ** 6))
print("số luỹ thừa của 100 không vượt 10^6:", sum(1 for k in range(30) if 100 ** k <= 10 ** 6))

[262144, 524288, 1048576] -> 2^19 = 524288 <= 10^6 < 2^20 = 1048576
số luỹ thừa của 2 không vượt 10^6: 20
số luỹ thừa của 100 không vượt 10^6: 4


### 3.2 Cơ số `1`: bẫy vòng lặp vô hạn

`1^k = 1` với mọi `k`. Vòng `while p <= bound: p *= 1` không bao giờ thoát. Cần xử lý riêng: với cơ số `1` chỉ có **một** giá trị `1`.

In [3]:
def powers(base, bound):
    """Các luỹ thừa base^k <= bound (k >= 0), tăng dần; cơ số 1 chỉ có [1]."""
    if base == 1:
        return [1] if bound >= 1 else []
    out, p = [], 1
    while p <= bound:
        out.append(p)
        p *= base
    return out


print(powers(2, 10), powers(3, 10), powers(1, 10), powers(1, 0), powers(100, 10 ** 6))

[1, 2, 4, 8] [1, 3, 9] [1] [] [1, 100, 10000, 1000000]


### 3.3 Tập hợp để khử trùng

Cùng một giá trị có thể sinh từ nhiều cặp `(i, j)` (ví dụ `x = 2, y = 4`: `2^2 + 4^0 = 5` và `2^0 + 4^1 = 5`). `set` loại trùng; hoặc `bytearray` đánh dấu theo chỉ số giá trị (vì `bound ≤ 10^6`).

In [4]:
print(sorted({a + b for a in powers(2, 20) for b in powers(4, 20)}))
print("2^2 + 4^0 =", 2 ** 2 + 4 ** 0, "| 2^0 + 4^1 =", 2 ** 0 + 4 ** 1)

[2, 3, 5, 6, 8, 9, 12, 17, 18, 20, 24, 32]
2^2 + 4^0 = 5 | 2^0 + 4^1 = 5


## 4. Ý tưởng nền tảng / trực giác

`x = 2, y = 3, bound = 10`: luỹ thừa của `2`: `[1, 2, 4, 8]`; luỹ thừa của `3`: `[1, 3, 9]`. Mọi tổng `a + b` với `a` trong danh sách đầu, `b` trong danh sách sau, giữ những tổng `≤ 10`:

| a \ b | 1 | 3 | 9 |
|---|---|---|---|
| 1 | 2 | 4 | 10 |
| 2 | 3 | 5 | 11 ✗ |
| 4 | 5 | 7 | 13 ✗ |
| 8 | 9 | 11 ✗ | 17 ✗ |

Tập các tổng hợp lệ `{2, 4, 10, 3, 5, 7, 9}` ⇒ sắp xếp: `[2,3,4,5,7,9,10]`. Số cặp tối đa `20 × 20 = 400` — rất nhỏ.

In [5]:
xs, ys = powers(2, 10), powers(3, 10)
tbl = {(a, b): a + b for a in xs for b in ys}
print(xs, ys)
print(sorted({v for v in tbl.values() if v <= 10}))

[1, 2, 4, 8] [1, 3, 9]
[2, 3, 4, 5, 7, 9, 10]


## 5. Brute force — kiểm tra từng giá trị `v`

**Ý tưởng:** với mỗi `v` từ `0` đến `bound`, kiểm tra `v` có phải số mạnh không: thử mọi `j` với `y^j <= v` và xem `v - y^j` có là luỹ thừa của `x` không (chia liên tục bằng `x`; riêng `x = 1` chỉ khi hiệu bằng `1`). Độ phức tạp `O(bound · log)`; dùng được cho `bound` nhỏ và là **oracle độc lập** (không dựng tập tổng).

In [6]:
def is_power_of(base, n):
    """n có phải base^k (k >= 0) không?"""
    if n < 1:
        return False
    if base == 1:
        return n == 1
    while n % base == 0:
        n //= base
    return n == 1


def is_powerful(x, y, v):
    """v = x^i + y^j với i, j >= 0?"""
    if y == 1:
        return is_power_of(x, v - 1)
    p = 1
    while p <= v:
        if is_power_of(x, v - p):
            return True
        p *= y
    return False


def powerful_brute(x, y, bound):
    return [v for v in range(bound + 1) if is_powerful(x, y, v)]


for (x, y, b), want in examples:
    assert powerful_brute(x, y, b) == want
print([powerful_brute(*args) for args, _ in examples])

[[2, 3, 4, 5, 7, 9, 10], [2, 4, 6, 8, 10, 14]]


In [7]:
# Code nộp LeetCode cho Brute force (chậm với bound lớn; để minh hoạ)
class SolutionBrute:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        def is_pow(base, n):
            if n < 1:
                return False
            if base == 1:
                return n == 1
            while n % base == 0:
                n //= base
            return n == 1

        res = []
        for v in range(bound + 1):
            if y == 1:
                ok = is_pow(x, v - 1)
            else:
                ok, p = False, 1
                while p <= v and not ok:
                    ok = is_pow(x, v - p)
                    p *= y
            if ok:
                res.append(v)
        return res

**Độ phức tạp:** `O(bound · log bound)` thời gian, `O(1)` bộ nhớ phụ. Với `bound = 10^6` vẫn chạy được trong vài giây nhưng lãng phí: chỉ có ≤ 441 đáp án.

**Nút thắt:** ta duyệt mọi `v` trong khi đáp án chỉ gồm vài trăm tổng. Hãy **sinh** các tổng thay vì **kiểm tra** từng `v`.

## 6. Bước đột phá

Có tối đa `~20` luỹ thừa mỗi cơ số ⇒ duyệt mọi cặp `(x^i, y^j)` (`≤ 441` cặp), giữ các tổng `≤ bound`, khử trùng bằng `set`. Chỉ cần xử lý riêng cơ số `1`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** hai vòng lặp + `set` (đáp án chính), **Cách 3** `bytearray` đánh dấu, **Cách 4** `bisect` trên luỹ thừa của `y`, **Cách 5** heap sinh theo thứ tự tăng.

### Cách 2 — Hai vòng lặp trên các luỹ thừa + `set`

**Giải thích từng đoạn:**

1. `xs = powers(x, bound)`, `ys = powers(y, bound)` (có xử lý cơ số `1`).
2. Với mọi `a ∈ xs`, `b ∈ ys`: nếu `a + b <= bound` thì thêm vào `set`.
3. Trả về `list(set)`; khi so sánh ta `sorted` (đề cho phép thứ tự bất kỳ).

In [8]:
def powerful_set(x, y, bound):
    xs, ys = powers(x, bound), powers(y, bound)
    res = set()
    for a in xs:
        for b in ys:
            if a + b <= bound:
                res.add(a + b)
    return list(res)


for (x, y, b), want in examples:
    assert sorted(powerful_set(x, y, b)) == want
print([sorted(powerful_set(*args)) for args, _ in examples])

[[2, 3, 4, 5, 7, 9, 10], [2, 4, 6, 8, 10, 14]]


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionSet:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        def powers(base):
            if base == 1:
                return [1] if bound >= 1 else []
            out, p = [], 1
            while p <= bound:
                out.append(p)
                p *= base
            return out

        res = set()
        for a in powers(x):
            for b in powers(y):
                if a + b <= bound:
                    res.add(a + b)
        return list(res)

**Độ phức tạp:** `O(log_x(bound) · log_y(bound))` thời gian (≤ 441), `O(đáp án)` bộ nhớ.

### Cách 3 — `bytearray` đánh dấu

**Ý tưởng:** tạo `seen = bytearray(bound + 1)`; đánh dấu `seen[a + b] = 1` với mỗi tổng hợp lệ; đáp án là các chỉ số được đánh dấu (tự động **tăng dần**, khử trùng). Chi phí bộ nhớ `O(bound)` nhưng tránh băm.

In [10]:
def powerful_marks(x, y, bound):
    seen = bytearray(bound + 1)
    for a in powers(x, bound):
        for b in powers(y, bound):
            s = a + b
            if s <= bound:
                seen[s] = 1
    return [v for v in range(bound + 1) if seen[v]]


for (x, y, b), want in examples:
    assert powerful_marks(x, y, b) == want
print([powerful_marks(*args) for args, _ in examples])

[[2, 3, 4, 5, 7, 9, 10], [2, 4, 6, 8, 10, 14]]


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionMarks:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        def powers(base):
            if base == 1:
                return [1] if bound >= 1 else []
            out, p = [], 1
            while p <= bound:
                out.append(p)
                p *= base
            return out

        seen = bytearray(bound + 1)
        ys = powers(y)
        for a in powers(x):
            for b in ys:
                if a + b <= bound:
                    seen[a + b] = 1
        return [v for v in range(bound + 1) if seen[v]]

**Độ phức tạp:** `O(441 + bound)` thời gian (bước quét `bytearray`), `O(bound)` bộ nhớ.

### Cách 4 — `bisect` trên luỹ thừa của `y`

**Ý tưởng:** `ys` đã tăng dần. Với mỗi `a ∈ xs`, các `b` thoả `a + b <= bound` là một **tiền tố** của `ys`: chiều dài `k = bisect_right(ys, bound - a)`. Thêm `a + b` cho `b` trong `ys[:k]` mà không cần `if` trong vòng trong.

In [12]:
def powerful_bisect(x, y, bound):
    xs, ys = powers(x, bound), powers(y, bound)
    res = set()
    for a in xs:
        k = bisect.bisect_right(ys, bound - a)          # số phần tử b trong ys thoả b <= bound - a
        res.update(a + b for b in ys[:k])
    return list(res)


for (x, y, b), want in examples:
    assert sorted(powerful_bisect(x, y, b)) == want
print([sorted(powerful_bisect(*args)) for args, _ in examples])

[[2, 3, 4, 5, 7, 9, 10], [2, 4, 6, 8, 10, 14]]


In [13]:
# Code nộp LeetCode cho Cách 4
import bisect


class SolutionBisect:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        def powers(base):
            if base == 1:
                return [1] if bound >= 1 else []
            out, p = [], 1
            while p <= bound:
                out.append(p)
                p *= base
            return out

        xs, ys = powers(x), powers(y)
        res = set()
        for a in xs:
            k = bisect.bisect_right(ys, bound - a)
            for b in ys[:k]:
                res.add(a + b)
        return list(res)

**Độ phức tạp:** `O(|xs| · log|ys| + số tổng)`; thực tế cùng bậc với Cách 2.

### Cách 5 — Heap sinh theo thứ tự tăng

**Ý tưởng:** trộn các dãy đã sắp `a_i + ys[j]` (với mỗi `i`, dãy tăng theo `j`) bằng min-heap `(tổng, i, j)`. Lấy tổng nhỏ nhất, nếu khác tổng vừa lấy thì ghi nhận; đẩy `(a_i + ys[j+1], i, j+1)`. Kết quả **đã sắp xếp**, không cần `set`.

In [14]:
def powerful_heap(x, y, bound):
    xs, ys = powers(x, bound), powers(y, bound)
    heap = [(a + ys[0], i, 0) for i, a in enumerate(xs) if ys and a + ys[0] <= bound]
    heapq.heapify(heap)
    res = []
    while heap:
        s, i, j = heapq.heappop(heap)
        if not res or res[-1] != s:                   # khử trùng: các tổng bằng nhau đứng cạnh nhau
            res.append(s)
        if j + 1 < len(ys) and xs[i] + ys[j + 1] <= bound:
            heapq.heappush(heap, (xs[i] + ys[j + 1], i, j + 1))
    return res


for (x, y, b), want in examples:
    assert powerful_heap(x, y, b) == want
print([powerful_heap(*args) for args, _ in examples])

[[2, 3, 4, 5, 7, 9, 10], [2, 4, 6, 8, 10, 14]]


In [15]:
# Code nộp LeetCode cho Cách 5
import heapq


class SolutionHeap:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        def powers(base):
            if base == 1:
                return [1] if bound >= 1 else []
            out, p = [], 1
            while p <= bound:
                out.append(p)
                p *= base
            return out

        xs, ys = powers(x), powers(y)
        heap = [(a + ys[0], i, 0) for i, a in enumerate(xs) if ys and a + ys[0] <= bound]
        heapq.heapify(heap)
        res = []
        while heap:
            s, i, j = heapq.heappop(heap)
            if not res or res[-1] != s:
                res.append(s)
            if j + 1 < len(ys) and xs[i] + ys[j + 1] <= bound:
                heapq.heappush(heap, (xs[i] + ys[j + 1], i, j + 1))
        return res

**Độ phức tạp:** `O(m · log m)` với `m ≤ 441` số tổng; `O(m)` bộ nhớ. Hữu ích khi cần **k-th nhỏ nhất** (mục 15).

## 8. TRACE từng bước

**Trace Cách 2** với `x = 2, y = 3, bound = 10`. Hàm trace in từng cặp `(a, b)`, tổng và quyết định:

In [16]:
def trace_set(x, y, bound):
    xs, ys = powers(x, bound), powers(y, bound)
    print(f"x={x}, y={y}, bound={bound}: xs={xs}, ys={ys}")
    res = set()
    for a in xs:
        for b in ys:
            s = a + b
            if s <= bound:
                new = s not in res
                res.add(s)
                print(f"  {a} + {b} = {s:3d} <= {bound}: {'thêm' if new else 'trùng'}")
            else:
                print(f"  {a} + {b} = {s:3d} >  {bound}: bỏ")
    out = sorted(res)
    print("=>", out)
    return out


assert trace_set(2, 3, 10) == [2, 3, 4, 5, 7, 9, 10]

x=2, y=3, bound=10: xs=[1, 2, 4, 8], ys=[1, 3, 9]
  1 + 1 =   2 <= 10: thêm
  1 + 3 =   4 <= 10: thêm
  1 + 9 =  10 <= 10: thêm
  2 + 1 =   3 <= 10: thêm
  2 + 3 =   5 <= 10: thêm
  2 + 9 =  11 >  10: bỏ
  4 + 1 =   5 <= 10: trùng
  4 + 3 =   7 <= 10: thêm
  4 + 9 =  13 >  10: bỏ
  8 + 1 =   9 <= 10: thêm
  8 + 3 =  11 >  10: bỏ
  8 + 9 =  17 >  10: bỏ
=> [2, 3, 4, 5, 7, 9, 10]


**Trace trường hợp trùng** `x = 2, y = 4, bound = 20`: `2^2 + 4^0 = 5` và `2^0 + 4^1 = 5` cho cùng một giá trị; `set` giữ một bản.

In [17]:
assert trace_set(2, 4, 20) == sorted(set(powerful_brute(2, 4, 20)))

x=2, y=4, bound=20: xs=[1, 2, 4, 8, 16], ys=[1, 4, 16]
  1 + 1 =   2 <= 20: thêm
  1 + 4 =   5 <= 20: thêm
  1 + 16 =  17 <= 20: thêm
  2 + 1 =   3 <= 20: thêm
  2 + 4 =   6 <= 20: thêm
  2 + 16 =  18 <= 20: thêm
  4 + 1 =   5 <= 20: trùng
  4 + 4 =   8 <= 20: thêm
  4 + 16 =  20 <= 20: thêm
  8 + 1 =   9 <= 20: thêm
  8 + 4 =  12 <= 20: thêm
  8 + 16 =  24 >  20: bỏ
  16 + 1 =  17 <= 20: trùng
  16 + 4 =  20 <= 20: trùng
  16 + 16 =  32 >  20: bỏ
=> [2, 3, 5, 6, 8, 9, 12, 17, 18, 20]


**Trace trường hợp cơ số 1** `x = 1, y = 2, bound = 10`: `xs = [1]`, `ys = [1, 2, 4, 8]` ⇒ tổng `2, 3, 5, 9`.

In [18]:
assert trace_set(1, 2, 10) == [2, 3, 5, 9]

x=1, y=2, bound=10: xs=[1], ys=[1, 2, 4, 8]
  1 + 1 =   2 <= 10: thêm
  1 + 2 =   3 <= 10: thêm
  1 + 4 =   5 <= 10: thêm
  1 + 8 =   9 <= 10: thêm
=> [2, 3, 5, 9]


## 9. Code nộp LeetCode

Chọn **Cách 2** (hai vòng lặp + `set`). Giữ nguyên `class Solution`, `powerfulIntegers(self, x, y, bound)`:

In [19]:
class Solution:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        def powers(base):
            if base == 1:                      # 1^k = 1: tránh vòng lặp vô hạn
                return [1] if bound >= 1 else []
            out, p = [], 1
            while p <= bound:
                out.append(p)
                p *= base
            return out

        res = set()
        for a in powers(x):
            for b in powers(y):
                if a + b <= bound:
                    res.add(a + b)             # set khử trùng
        return list(res)


print([sorted(Solution().powerfulIntegers(*args)) for args, _ in examples])
assert all(sorted(Solution().powerfulIntegers(*args)) == want for args, want in examples)

[[2, 3, 4, 5, 7, 9, 10], [2, 4, 6, 8, 10, 14]]


## 10. Vì sao đúng

**(a) Mọi số mạnh `≤ bound` được sinh.** Nếu `v = x^i + y^j ≤ bound` thì vì mọi hạng tử `≥ 1`, `x^i ≤ v - 1 < bound` và `y^j ≤ bound`: cả hai luỹ thừa nằm trong `xs`, `ys` (các luỹ thừa `≤ bound`). Vòng lặp kép gặp cặp `(x^i, y^j)` và ghi `v`.

**(b) Không sinh thừa.** Mỗi phần tử ghi vào có dạng `a + b` với `a = x^i ∈ xs`, `b = y^j ∈ ys` và `a + b ≤ bound`: chính là số mạnh `≤ bound`.

**(c) Hữu hạn và đủ.** Với cơ số `b ≥ 2`, `b^k` tăng nghiêm ngặt nên `powers` dừng sau `⌊log_b bound⌋ + 1` phần tử. Với `b = 1`, mọi `1^k = 1` nên tập các **giá trị** luỹ thừa chỉ là `{1}`; `powers` trả `[1]` (nếu `bound ≥ 1`), không lặp vô hạn. Nếu `bound = 0` thì không có luỹ thừa nào `≤ 0` ⇒ kết quả rỗng; nếu `bound = 1` thì tổng nhỏ nhất `1 + 1 = 2 > 1` ⇒ rỗng.

**(d) Khử trùng.** `set`/`bytearray` đảm bảo mỗi giá trị một lần (đề yêu cầu). Cách 5 khử trùng nhờ thứ tự tăng: các tổng bằng nhau đứng cạnh nhau trong luồng ra của heap.

**(e) Cách 4.** `ys` tăng ⇒ `{b ∈ ys : b ≤ bound − a}` là tiền tố; `bisect_right(ys, bound - a)` là độ dài tiền tố đó.

**(f) Cách 5.** Mỗi dãy `a_i + ys[0..]` tăng nghiêm ngặt; heap luôn chứa phần tử nhỏ nhất chưa ghi của mỗi dãy ⇒ rút ra theo thứ tự không giảm (thuật toán trộn k dãy đã sắp).

**(g) Kiểm chứng bằng máy.** Mọi cách khớp oracle **kiểm tra từng giá trị** (`is_powerful`) trên toàn bộ `x, y ∈ [1, 20]` và `bound ∈ [0, 300]` (mục 12D), và khớp oracle "luỹ thừa với số mũ ≤ 20" cho `x, y ≤ 100`, `bound ≤ 10^6` (mục 12E).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Kiểm tra từng `v` | `O(bound · log)` | `O(1)` | oracle, `bound` nhỏ |
| 2. Hai vòng lặp + `set` | `O(log_x b · log_y b)` | `O(đáp án)` | **đáp án chính** |
| 3. `bytearray` đánh dấu | `O(441 + bound)` | `O(bound)` | kết quả đã sắp |
| 4. `bisect` | `O(\|xs\| log \|ys\| + m)` | `O(đáp án)` | bỏ `if` vòng trong |
| 5. Heap | `O(m log m)` | `O(m)` | kết quả đã sắp, k-th nhỏ nhất |

Đo thực tế (`bound = 10^6`):

In [20]:
def bench(fn, args, repeat=30):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


for name, fn in [("set", powerful_set), ("marks", powerful_marks), ("bisect", powerful_bisect), ("heap", powerful_heap)]:
    print(f"{name:7s}: x=2,y=3,bound=1e6: {bench(fn, (2, 3, 10 ** 6)) * 1e3:7.3f} ms")
print(f"brute  : x=2,y=3,bound=3e4: {bench(powerful_brute, (2, 3, 30000), repeat=1) * 1e3:7.1f} ms")

set    : x=2,y=3,bound=1e6:   0.021 ms


marks  : x=2,y=3,bound=1e6:  30.079 ms
bisect : x=2,y=3,bound=1e6:   0.022 ms
heap   : x=2,y=3,bound=1e6:   0.089 ms
brute  : x=2,y=3,bound=3e4:    51.4 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn `x, y ∈ [1, 20]`, `bound ∈ [0, 300]`, (E) tính chất / ngẫu nhiên tới giới hạn đề (seed cố định), (F) giới hạn.

Thứ tự kết quả không quan trọng: hàm kiểm tra `norm(res)` kiểm tra không trùng rồi `sorted`.

### A. Ví dụ của đề

In [21]:
def norm(res):
    assert len(res) == len(set(res)), "kết quả có giá trị trùng"
    return sorted(res)


IMPLS = {
    "set": powerful_set,
    "marks": powerful_marks,
    "bisect": powerful_bisect,
    "heap": powerful_heap,
    "brute": powerful_brute,
    "Solution": lambda x, y, b: Solution().powerfulIntegers(x, y, b),
    "SolutionSet": lambda x, y, b: SolutionSet().powerfulIntegers(x, y, b),
    "SolutionMarks": lambda x, y, b: SolutionMarks().powerfulIntegers(x, y, b),
    "SolutionBisect": lambda x, y, b: SolutionBisect().powerfulIntegers(x, y, b),
    "SolutionHeap": lambda x, y, b: SolutionHeap().powerfulIntegers(x, y, b),
    "SolutionBrute": lambda x, y, b: SolutionBrute().powerfulIntegers(x, y, b),
}
SLOW = {"brute", "SolutionBrute"}


def check(x, y, bound, want=None, names=None, include_slow=True):
    if want is None:
        want = oracle_small(x, y, bound) if bound <= 2000 else oracle_pow(x, y, bound)
    for name in (names or IMPLS):
        if name in SLOW and (not include_slow or bound > 3000):
            continue
        got = norm(IMPLS[name](x, y, bound))
        assert got == want, (name, x, y, bound, got, want)


def oracle_small(x, y, bound):
    return [v for v in range(bound + 1) if is_powerful(x, y, v)]


def oracle_pow(x, y, bound):
    """Oracle độc lập với powers(): mọi số mũ 0..20 (đủ vì 2^20 > 10^6), không cần xử lý riêng cơ số 1."""
    return sorted({x ** i + y ** j for i in range(21) for j in range(21) if x ** i + y ** j <= bound})


for args, want in examples:
    assert oracle_small(*args) == want and oracle_pow(*args) == want
    check(*args, want=want)
print("ví dụ đề: hai oracle và", len(IMPLS), "cài đặt đúng")

ví dụ đề: hai oracle và 11 cài đặt đúng


### B. Edge case

- `bound = 0` và `bound = 1` ⇒ `[]` (tổng nhỏ nhất `1 + 1 = 2`); `bound = 2` ⇒ `[2]`.
- `x = 1`, `y = 1` (cả hai là cơ số `1`) — vòng lặp vô hạn nếu quên.
- `x = 1` hoặc `y = 1` một bên; `x = y`.
- `x = 100, y = 100, bound = 10^6`.
- `bound = 10^6` với `x = 2, y = 2`.
- Hai cơ số cho tổng trùng (`2` và `4`, `3` và `9`).
- Một cơ số lớn hơn `bound` (`x = 100, bound = 50`).

In [22]:
cases = [
    (2, 3, 0), (2, 3, 1), (2, 3, 2), (1, 1, 0), (1, 1, 1), (1, 1, 2), (1, 1, 10 ** 6), (1, 2, 10), (2, 1, 10), (1, 100, 10 ** 6),
    (100, 1, 10 ** 6), (2, 2, 100), (3, 3, 100), (2, 4, 100), (3, 9, 100), (100, 100, 50), (100, 100, 10 ** 6), (2, 2, 10 ** 6),
    (5, 7, 2), (2, 3, 3), (100, 2, 101), (7, 1, 8), (1, 7, 8),
]
for x, y, b in cases:
    check(x, y, b)
print(len(cases), "edge case: mọi cài đặt khớp oracle")
print("1,2,10 ->", powerful_set(1, 2, 10) and sorted(powerful_set(1, 2, 10)), "| 100,100,1e6 ->", sorted(powerful_set(100, 100, 10 ** 6)))

23 edge case: mọi cài đặt khớp oracle
1,2,10 -> [2, 3, 5, 9] | 100,100,1e6 -> [2, 101, 200, 10001, 10100, 20000]


### C. Phản ví dụ: cài đặt sai phải bị bắt

Các hàm "bug" có **giới hạn số vòng** để một vòng lặp vô hạn không treo notebook (ném `RuntimeError`, được coi là sai).

In [23]:
def capped_powers(base, bound, cap=60):
    out, p, steps = [], 1, 0
    while p <= bound:
        out.append(p)
        p *= base
        steps += 1
        if steps > cap:
            raise RuntimeError("vòng lặp vô hạn (cơ số 1)")
    return out


def bug_base_one_loop(x, y, bound):                  # quên xử lý cơ số 1
    res = set()
    for a in capped_powers(x, bound):
        for b in capped_powers(y, bound):
            if a + b <= bound:
                res.add(a + b)
    return list(res)


def bug_strict_less(x, y, bound):                    # dùng < thay <=
    res = set()
    for a in powers(x, bound):
        for b in powers(y, bound):
            if a + b < bound:
                res.add(a + b)
    return list(res)


def bug_exponent_from_one(x, y, bound):              # bắt đầu số mũ từ 1 (bỏ x^0, y^0)
    res = set()
    for a in powers(x, bound)[1:]:
        for b in powers(y, bound)[1:]:
            if a + b <= bound:
                res.add(a + b)
    return list(res)


def bug_duplicates(x, y, bound):                     # dùng list thay set => trùng giá trị
    res = []
    for a in powers(x, bound):
        for b in powers(y, bound):
            if a + b <= bound:
                res.append(a + b)
    return res


def bug_break_outer(x, y, bound):                    # break nhầm vòng ngoài khi tổng vượt bound
    res = set()
    for a in powers(x, bound):
        for b in powers(y, bound):
            if a + b > bound:
                return sorted(res)
            res.add(a + b)
    return sorted(res)


def bug_bound_minus_one_exponent(x, y, bound):       # chỉ xét luỹ thừa < bound/2 (cắt quá sớm)
    res = set()
    for a in [p for p in powers(x, bound) if p <= bound // 2]:
        for b in [p for p in powers(y, bound) if p <= bound // 2]:
            if a + b <= bound:
                res.add(a + b)
    return list(res)


BUGS = [bug_base_one_loop, bug_strict_less, bug_exponent_from_one, bug_duplicates, bug_break_outer, bug_bound_minus_one_exponent]
for bug in BUGS:
    found = None
    for x in range(1, 7):
        for y in range(1, 7):
            for b in range(0, 40):
                try:
                    got = sorted(bug(x, y, b))
                    if len(got) != len(set(got)):
                        got = "có trùng"
                except RuntimeError as e:
                    got = "RuntimeError"
                if got != oracle_pow(x, y, b):
                    found = (x, y, b, got)
                    break
            if found:
                break
        if found:
            break
    assert found, bug.__name__
    print(f"{bug.__name__:32s} sai trên x={found[0]}, y={found[1]}, bound={found[2]}: ra {found[3]}, đúng {oracle_pow(*found[:3])}")

bug_base_one_loop                sai trên x=1, y=1, bound=1: ra RuntimeError, đúng []
bug_strict_less                  sai trên x=1, y=1, bound=2: ra [], đúng [2]
bug_exponent_from_one            sai trên x=1, y=1, bound=2: ra [], đúng [2]
bug_duplicates                   sai trên x=2, y=2, bound=3: ra có trùng, đúng [2, 3]
bug_break_outer                  sai trên x=2, y=2, bound=4: ra [2, 3], đúng [2, 3, 4]
bug_bound_minus_one_exponent     sai trên x=1, y=2, bound=3: ra [2], đúng [2, 3]


### D. Vét cạn `x, y ∈ [1, 20]`, `bound ∈ [0, 300]`

Với mỗi cặp `(x, y)` ta tính oracle `is_powerful` (kiểm tra từng giá trị, độc lập với việc sinh tổng) cho `bound = 300`, rồi với **mọi** `bound ∈ [0, 300]` lọc `v ≤ bound` để được đáp án mong đợi và so với mọi cài đặt (trừ cài đặt brute cho phần lớn `bound` để tiết kiệm thời gian — brute vẫn được thử đầy đủ cho `bound ≤ 40`). Tổng `20 × 20 × 301 = 120 400` ca.

In [24]:
t0 = time.perf_counter()
cnt = 0
fast = [n for n in IMPLS if n not in SLOW]
for x in range(1, 21):
    for y in range(1, 21):
        full = oracle_small(x, y, 300)
        assert full == oracle_pow(x, y, 300)                     # hai oracle độc lập khớp nhau
        for b in range(0, 301):
            want = [v for v in full if v <= b]
            for name in fast:
                assert norm(IMPLS[name](x, y, b)) == want, (name, x, y, b)
            if b <= 40:
                for name in SLOW:
                    assert norm(IMPLS[name](x, y, b)) == want, (name, x, y, b)
            cnt += 1
print(cnt, "ca (x, y ∈ [1,20], bound ∈ [0,300]): hai oracle khớp nhau,", len(fast), "cài đặt nhanh khớp;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 120400

120400 ca (x, y ∈ [1,20], bound ∈ [0,300]): hai oracle khớp nhau, 9 cài đặt nhanh khớp; 4.8 s


### E. Tính chất / ngẫu nhiên tới giới hạn đề

Seed cố định. Kiểm tra (cài đặt `bytearray` chỉ thử khi `bound ≤ 50 000` vì tốn `O(bound)`): (1) khớp `oracle_pow`; (2) mọi giá trị `≤ bound` và thật sự có dạng `x^i + y^j` (kiểm bằng `is_powerful`); (3) không trùng; (4) số lượng `≤ 441`; (5) đối xứng `x ↔ y`; (6) đơn điệu theo `bound` (`bound` lớn hơn cho tập chứa tập cũ); (7) kết quả không rỗng ⇔ `bound ≥ 2`.

In [25]:
def names_for(b):
    """Cài đặt bytearray tốn O(bound) nên chỉ thử khi bound không quá lớn."""
    return [n for n in fast if b <= 50000 or n not in ("marks", "SolutionMarks")]


rng = random.Random(20240970)
t0 = time.perf_counter()
for it in range(1500):
    x, y = rng.randint(1, 100), rng.randint(1, 100)
    if it % 3 == 0:
        x, y = rng.randint(1, 6), rng.randint(1, 6)                 # nhiều số mũ hơn
    b = rng.choice([rng.randint(0, 10 ** 6), rng.randint(0, 1000), 10 ** 6])
    want = oracle_pow(x, y, b)
    for name in names_for(b):
        assert norm(IMPLS[name](x, y, b)) == want, (name, x, y, b)
    res = Solution().powerfulIntegers(x, y, b)
    assert len(res) == len(set(res)) <= 441 and all(v <= b for v in res)                       # (2) (3) (4)
    for v in rng.sample(res, min(len(res), 5)):
        assert is_powerful(x, y, v), (x, y, v)                                                   # (2)
    assert sorted(Solution().powerfulIntegers(y, x, b)) == sorted(res)                           # (5)
    b2 = rng.randint(0, 10 ** 6)
    lo, hi = sorted((b, b2))
    assert set(Solution().powerfulIntegers(x, y, lo)) <= set(Solution().powerfulIntegers(x, y, hi))   # (6)
    assert (len(res) > 0) == (b >= 2)                                                            # (7)
print("1500 ca ngẫu nhiên tới 10^6: 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

1500 ca ngẫu nhiên tới 10^6: 7 tính chất đúng; 0.5 s


### F. Giới hạn

`bound = 10^6` với mọi cặp cơ số "nhỏ" (`2..6`), các cặp lớn (`100`), `x = 1`/`y = 1` với `bound = 10^6`; kiểm thời gian tổng.

In [26]:
t0 = time.perf_counter()
for x in list(range(1, 7)) + [10, 50, 99, 100]:
    for y in list(range(1, 7)) + [10, 50, 99, 100]:
        want = oracle_pow(x, y, 10 ** 6)
        for name in names_for(10 ** 6):
            assert norm(IMPLS[name](x, y, 10 ** 6)) == want, (name, x, y)
print("bound = 10^6 với 10 × 10 cặp cơ số: mọi cài đặt đúng;", round(time.perf_counter() - t0, 2), "s")
print("x=2,y=2,bound=1e6: số đáp án", len(Solution().powerfulIntegers(2, 2, 10 ** 6)), "| x=2,y=3:", len(Solution().powerfulIntegers(2, 3, 10 ** 6)))

bound = 10^6 với 10 × 10 cặp cơ số: mọi cài đặt đúng; 0.03 s
x=2,y=2,bound=1e6: số đáp án 209 | x=2,y=3: 254


## 13. Các lỗi người mới hay gặp

### Lỗi 1: vòng lặp vô hạn khi cơ số bằng `1`

In [27]:
try:
    bug_base_one_loop(1, 5, 100)
    print("không lỗi?")
except RuntimeError as e:
    print("bug:", e)
print("đúng:", sorted(Solution().powerfulIntegers(1, 5, 100)))

bug: vòng lặp vô hạn (cơ số 1)
đúng: [2, 6, 26]


**Vì sao sai:** `p *= 1` không đổi `p` nên `p <= bound` luôn đúng. **Đúng:** nếu cơ số bằng `1` thì chỉ có một luỹ thừa `1`.

### Lỗi 2: dùng `<` thay `<=`

In [28]:
print("bug:", sorted(bug_strict_less(2, 3, 10)), "| đúng:", sorted(Solution().powerfulIntegers(2, 3, 10)))
assert 10 not in bug_strict_less(2, 3, 10) and 10 in Solution().powerfulIntegers(2, 3, 10)

bug: [2, 3, 4, 5, 7, 9] | đúng: [2, 3, 4, 5, 7, 9, 10]


**Vì sao sai:** "giá trị nhỏ hơn **hoặc bằng** `bound`" ⇒ `<=`.

### Lỗi 3: bắt đầu số mũ từ 1 (bỏ `x^0`, `y^0`)

In [29]:
print("bug:", sorted(bug_exponent_from_one(2, 3, 10)), "| đúng:", sorted(Solution().powerfulIntegers(2, 3, 10)))
assert 2 not in bug_exponent_from_one(2, 3, 10)

bug: [5, 7] | đúng: [2, 3, 4, 5, 7, 9, 10]


**Vì sao sai:** đề cho `i >= 0`, `j >= 0` nên `x^0 = 1` phải được xét (số `2 = 1 + 1`). **Đúng:** `powers` bắt đầu từ `p = 1`.

### Lỗi 4: trả về giá trị trùng

In [30]:
res = bug_duplicates(2, 4, 20)
print("bug:", sorted(res), "| đúng:", sorted(Solution().powerfulIntegers(2, 4, 20)))
assert len(res) != len(set(res))

bug: [2, 3, 5, 5, 6, 8, 9, 12, 17, 17, 18, 20, 20] | đúng: [2, 3, 5, 6, 8, 9, 12, 17, 18, 20]


**Vì sao sai:** đề: "mỗi giá trị xuất hiện tối đa một lần". **Đúng:** dùng `set`.

### Lỗi 5: `break`/`return` nhầm vòng ngoài

In [31]:
print("bug:", bug_break_outer(2, 3, 10), "| đúng:", sorted(Solution().powerfulIntegers(2, 3, 10)))
assert bug_break_outer(2, 3, 10) != sorted(Solution().powerfulIntegers(2, 3, 10))

bug: [2, 3, 4, 5, 10] | đúng: [2, 3, 4, 5, 7, 9, 10]


**Vì sao sai:** khi `a + b > bound` chỉ nên bỏ qua **các `b` lớn hơn** của cùng `a` (vòng trong), không dừng toàn bộ. **Đúng:** `break` vòng trong hoặc `continue`.

### Lỗi 6 (bonus): cắt luỹ thừa quá sớm

In [32]:
print("bug:", sorted(bug_bound_minus_one_exponent(2, 3, 10)), "| đúng:", sorted(Solution().powerfulIntegers(2, 3, 10)))
assert sorted(bug_bound_minus_one_exponent(2, 3, 10)) != sorted(Solution().powerfulIntegers(2, 3, 10))

bug: [2, 3, 4, 5, 7] | đúng: [2, 3, 4, 5, 7, 9, 10]


**Vì sao sai:** một số hạng có thể lớn gần `bound − 1` (hạng còn lại là `1`): `9 + 1 = 10`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tập các số dạng `x^i + y^j` (hoặc tổng hai dãy tăng nhanh) dưới một ngưỡng" → số phần tử mỗi dãy rất nhỏ (`log`) ⇒ **liệt kê toàn bộ cặp + `set`**.

**Mẹo:**

1. Luôn xử lý riêng cơ số `1` khi sinh luỹ thừa.
2. Mỗi dãy chỉ có `O(log bound)` phần tử ⇒ vòng lặp kép rẻ.
3. Khử trùng bằng `set`; nếu cần kết quả đã sắp thì `sorted` cuối hoặc dùng `bytearray`/heap.
4. Tự viết oracle **theo định nghĩa** (`is_powerful(v)`) cho `bound` nhỏ.
5. Cẩn thận `break` đúng vòng.

**Họ bài cùng pattern:** Ugly Number II (264), Super Ugly Number (313), Sum of Two Powers, Count Primes (204) (bảng đánh dấu), Four Divisors, Count Pairs With XOR in a Range.

## 15. Biến thể

1. **`bound` rất lớn** (tới `10^18`): chỉ đếm; mỗi dãy có `≤ 60` luỹ thừa ⇒ vẫn rẻ.
2. **Ba cơ số** `x^i + y^j + z^k`.
3. **Số mạnh nhỏ thứ `k`** (dùng heap, dừng sớm).

Cài đặt và đối chiếu vét cạn:

In [33]:
def count_powerful(x, y, bound):
    """Biến thể 1: đếm số số mạnh <= bound (bound có thể tới 10^18)."""
    return len({a + b for a in powers(x, bound) for b in powers(y, bound) if a + b <= bound})


def powerful_three(x, y, z, bound):
    """Biến thể 2: x^i + y^j + z^k <= bound."""
    return sorted({a + b + c for a in powers(x, bound) for b in powers(y, bound) for c in powers(z, bound) if a + b + c <= bound})


def kth_powerful(x, y, k, bound=10 ** 18):
    """Biến thể 3: số mạnh nhỏ thứ k (tính từ 1) bằng heap; None nếu không đủ."""
    xs, ys = powers(x, bound), powers(y, bound)
    heap = [(a + ys[0], i, 0) for i, a in enumerate(xs)]
    heapq.heapify(heap)
    last, cnt = None, 0
    while heap:
        s, i, j = heapq.heappop(heap)
        if s != last:
            last, cnt = s, cnt + 1
            if cnt == k:
                return s
        if j + 1 < len(ys):
            heapq.heappush(heap, (xs[i] + ys[j + 1], i, j + 1))
    return None


for x, y, b in [(2, 3, 10), (2, 3, 10 ** 6), (1, 5, 1000), (7, 7, 10 ** 5), (1, 1, 100)]:
    assert count_powerful(x, y, b) == len(oracle_pow(x, y, b))
print("count 2,3,10^18 =", count_powerful(2, 3, 10 ** 18), "(số mũ ≤ 60 nên rất nhanh)")
for _ in range(100):
    x, y, z = (random.Random(_).randint(1, 5) for _ in range(3))
    b = random.Random(_ + 1).randint(0, 500)
    brute3 = sorted({x ** i + y ** j + z ** k for i in range(11) for j in range(11) for k in range(11) if x ** i + y ** j + z ** k <= b})
    assert powerful_three(x, y, z, b) == brute3
for x, y in [(2, 3), (1, 2), (3, 3), (5, 2)]:
    lst = oracle_pow(x, y, 2000)
    for k in range(1, min(len(lst), 25) + 1):
        assert kth_powerful(x, y, k) == lst[k - 1], (x, y, k)
print("ba cơ số và k-th nhỏ nhất: khớp vét cạn")

count 2,3,10^18 = 2274 (số mũ ≤ 60 nên rất nhanh)
ba cơ số và k-th nhỏ nhất: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra từng `v` | `v - y^j` là luỹ thừa của `x`? | `O(bound·log)` | `O(1)` | oracle, `bound` nhỏ |
| 2. Hai vòng + `set` | sinh mọi tổng, khử trùng | `O(log²)` | `O(m)` | **đáp án chính** |
| 3. `bytearray` | đánh dấu theo chỉ số | `O(bound)` | `O(bound)` | cần kết quả đã sắp |
| 4. `bisect` | tiền tố của `ys` | `O(log² )` | `O(m)` | bỏ `if` vòng trong |
| 5. Heap | trộn các dãy đã sắp | `O(m log m)` | `O(m)` | k-th nhỏ nhất |

Tất cả khớp oracle `is_powerful` trên `120 400` ca (mục 12D) và khớp oracle số mũ ≤ 20 trên ca ngẫu nhiên tới `10^6`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy (sắp xếp kết quả):

In [34]:
print(sorted(Solution().powerfulIntegers(2, 1, 10)), sorted(Solution().powerfulIntegers(1, 1, 100)),
      sorted(Solution().powerfulIntegers(5, 7, 2)), sorted(Solution().powerfulIntegers(2, 3, 1)))

[2, 3, 5, 9] [2] [2] []


### Bài 2 (tính tay)

`x = 3, y = 5, bound = 15`. (a) Liệt kê `xs`, `ys`. (b) Liệt kê các tổng `≤ 15`. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm sinh luỹ thừa an toàn với cơ số `1`:

In [35]:
def powers_todo(base, bound):
    if base == 1:
        pass            # TODO 1: trả về [1] nếu bound >= 1, ngược lại []
    out, p = [], 1
    while p <= bound:
        pass            # TODO 2: thêm p vào out và nhân p với base
    return out


# Khi làm xong, bỏ comment để kiểm tra:
# assert powers_todo(2, 10) == [1, 2, 4, 8] and powers_todo(1, 10) == [1] and powers_todo(1, 0) == [] and powers_todo(100, 10 ** 6) == [1, 100, 10000, 1000000]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`bytearray`** (kết quả đã sắp):

In [36]:
class MySolution:
    def powerfulIntegers(self, x, y, bound):
        pass            # TODO


# Khi làm xong:
# assert all(sorted(MySolution().powerfulIntegers(*a)) == w for a, w in examples)

### Bài 5 (mini challenge): đếm với `bound` tới `10^18`

Viết `count_big(x, y, bound)` trả về **số lượng** số mạnh `≤ bound` với `bound` tới `10^18` (không dựng danh sách `bound` phần tử). Kiểm bằng `oracle_pow`-kiểu: dùng số mũ đủ lớn (≤ 60).

In [37]:
def count_big(x, y, bound):
    pass            # TODO


# Khi làm xong:
# assert count_big(2, 3, 10) == 7 and count_big(1, 1, 10 ** 18) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: cơ số `1` chỉ cho giá trị `1`.
- Mức 2: `(2, 1, 10)`: `2^i + 1` với `2^i ∈ {1,2,4,8}`.
- Mức 3: kết quả `[2, 3, 5, 9]`, `[2]`, `[2]` (với `bound = 2` chỉ có `1 + 1`), `[]` (với `bound = 1` tổng nhỏ nhất `2` đã vượt).

**Bài 2**

- Mức 1: luỹ thừa của `3` không vượt `15`: `1, 3, 9`.
- Mức 2: của `5`: `1, 5`.
- Mức 3: các tổng `≤ 15`: `1+1, 1+5, 3+1, 3+5, 9+1, 9+5` ⇒ `[2, 4, 6, 8, 10, 14]`.

**Bài 3**

- Mức 1: `return [1] if bound >= 1 else []`.
- Mức 2: `out.append(p); p *= base`.
- Mức 3: giống `powers` ở mục 3.

**Bài 4**

- Mức 1: `seen = bytearray(bound + 1)`.
- Mức 2: đánh dấu `seen[a + b] = 1` khi `a + b <= bound`.
- Mức 3: trả `[v for v in range(bound + 1) if seen[v]]`.

**Bài 5**

- Mức 1: không cần mảng `bound` phần tử: dùng `set` các tổng.
- Mức 2: mỗi cơ số `≥ 2` có `≤ 60` luỹ thừa `≤ 10^18`.
- Mức 3: `len({a + b for a in powers(x, bound) for b in powers(y, bound) if a + b <= bound})`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [38]:
vals = (sorted(Solution().powerfulIntegers(2, 1, 10)), sorted(Solution().powerfulIntegers(1, 1, 100)),
        sorted(Solution().powerfulIntegers(5, 7, 2)), sorted(Solution().powerfulIntegers(2, 3, 1)))
print(vals)
assert vals == ([2, 3, 5, 9], [2], [2], [])
print("Bài 1 OK")

([2, 3, 5, 9], [2], [2], [])
Bài 1 OK


### Bài 2

In [39]:
xs, ys = powers(3, 15), powers(5, 15)
sums = sorted({a + b for a in xs for b in ys if a + b <= 15})
print(xs, ys, sums)
assert xs == [1, 3, 9] and ys == [1, 5] and sums == [2, 4, 6, 8, 10, 14]
print("Bài 2 OK")

[1, 3, 9] [1, 5] [2, 4, 6, 8, 10, 14]
Bài 2 OK


### Bài 3

In [40]:
def powers_answer(base, bound):
    if base == 1:
        return [1] if bound >= 1 else []
    out, p = [], 1
    while p <= bound:
        out.append(p)
        p *= base
    return out


assert powers_answer(2, 10) == [1, 2, 4, 8] and powers_answer(1, 10) == [1] and powers_answer(1, 0) == [] and powers_answer(100, 10 ** 6) == [1, 100, 10000, 1000000]
for base in range(1, 30):
    for bound in (0, 1, 7, 100, 10 ** 6):
        assert powers_answer(base, bound) == powers(base, bound)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [41]:
class MySolutionAnswer:
    def powerfulIntegers(self, x, y, bound):
        def powers(base):
            if base == 1:
                return [1] if bound >= 1 else []
            out, p = [], 1
            while p <= bound:
                out.append(p)
                p *= base
            return out

        seen = bytearray(bound + 1)
        for a in powers(x):
            for b in powers(y):
                if a + b <= bound:
                    seen[a + b] = 1
        return [v for v in range(bound + 1) if seen[v]]


assert all(sorted(MySolutionAnswer().powerfulIntegers(*a)) == w for a, w in examples)
rng = random.Random(4)
for _ in range(100):
    x, y, b = rng.randint(1, 100), rng.randint(1, 100), rng.randint(0, 20000)
    assert MySolutionAnswer().powerfulIntegers(x, y, b) == oracle_pow(x, y, b)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [42]:
def count_big_answer(x, y, bound):
    return len({a + b for a in powers(x, bound) for b in powers(y, bound) if a + b <= bound})


assert count_big_answer(2, 3, 10) == 7 and count_big_answer(1, 1, 10 ** 18) == 1
rng = random.Random(5)
for _ in range(100):
    x, y, b = rng.randint(1, 100), rng.randint(1, 100), rng.randint(0, 10 ** 6)
    assert count_big_answer(x, y, b) == len(oracle_pow(x, y, b))
print("Bài 5 OK: count(2,3,10^18) =", count_big_answer(2, 3, 10 ** 18))

Bài 5 OK: count(2,3,10^18) = 2274


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tập tổng hai dãy luỹ thừa dưới ngưỡng → sinh mọi cặp + `set`):**

```python
class Solution:
    def powerfulIntegers(self, x: int, y: int, bound: int) -> List[int]:
        def powers(base):
            if base == 1:
                return [1] if bound >= 1 else []
            out, p = [], 1
            while p <= bound:
                out.append(p)
                p *= base
            return out

        return list({a + b for a in powers(x) for b in powers(y) if a + b <= bound})
```

**Checklist trước khi submit:**

- [ ] Đã xử lý cơ số `1` (tránh vòng lặp vô hạn)?
- [ ] Số mũ bắt đầu từ `0` (`x^0 = 1`)?
- [ ] Điều kiện `a + b <= bound` (không phải `<`)?
- [ ] Khử trùng (dùng `set`)?
- [ ] `bound = 0` hoặc `1` cho `[]`?
- [ ] Đã thử `(2,3,10)`, `(3,5,15)`, `(1,1,100)`, `(2,1,10)`?
- [ ] Đúng tên `class Solution`, `powerfulIntegers(self, x, y, bound)`, không in debug.